# 02. 학습과 비교 실험 (YOLOv8)

`01b_auto_label.ipynb` 셀 9가 Drive에 저장한 `dataset_split.zip`(또는 Roboflow에서 내보낸 데이터셋)으로 여러 설정을 학습하고, **테스트 세트** 기준 성능과 속도를 한 표로 비교합니다.

**시작 전 확인**
1. 런타임 → 런타임 유형 변경 → **T4 GPU**
2. Roboflow를 쓴다면 내보내기에서 **Resize를 끄거나 "Fit within 1280×1280"** 으로 설정했는지 (기본값 640이면 해상도 비교가 무의미해짐, `docs/labeling_guide.md` 참고)
3. Colab은 연결이 끊기면 파일이 사라지므로, 셀 1에서 Google Drive 저장을 켜 두는 것을 권장합니다.

In [ ]:
# ===== 셀 1: 환경 준비 =====
REPO_URL = "https://github.com/djkdb/beeguard-hornet-tracker.git"   # 본인 저장소 주소로 바꾸기
USE_DRIVE = True   # 학습 결과를 Google Drive에 저장 (연결이 끊겨도 보존)

!pip -q install ultralytics
import os, subprocess
if not os.path.isdir("/content/repo"):
    subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=False)
REPO = "/content/repo" if os.path.isdir("/content/repo") else "/content"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/beeguard/runs"
else:
    PROJECT = "/content/runs"
os.makedirs(PROJECT, exist_ok=True)
print("REPO:", REPO, "| 학습 결과 저장 위치:", PROJECT)

## 데이터셋 가져오기
셋 중 하나만 씁니다. 셀 2-C가 먼저 데이터셋을 풀면 2-A는 건너뜁니다.
- **방법 C**: `01b` 셀 9가 Drive에 저장한 `dataset_split.zip` (이번 제출에서 쓴 방법)
- **방법 A**: Roboflow에서 YOLOv8 형식 zip을 내려받아 업로드
- **방법 B**: Roboflow의 "Download code"에 나오는 코드를 붙여넣기 (본인 프로젝트, 본인 API 키)

In [ ]:
# ===== 셀 2-C: Drive의 dataset_split.zip (01b 셀 9에서 만든 것) =====
import zipfile, shutil
SPLIT_ZIP = "/content/drive/MyDrive/beeguard/dataset_split.zip"   # 셀 1에서 USE_DRIVE = True여야 보임
if os.path.exists(SPLIT_ZIP):
    shutil.rmtree("/content/dataset", ignore_errors=True)
    zipfile.ZipFile(SPLIT_ZIP).extractall("/content/dataset")
    print(os.listdir("/content/dataset"))
else:
    print("dataset_split.zip이 없습니다. 아래 셀 2-A나 2-B를 쓰세요.")

In [ ]:
# ===== 셀 2-A: zip 업로드 =====
from google.colab import files
import zipfile, shutil
if not os.path.isdir("/content/dataset/train"):    # 셀 2-C로 이미 풀었으면 건너뜀
    up = files.upload()                      # Roboflow에서 받은 zip 선택
    zip_name = next(iter(up))
    shutil.rmtree("/content/dataset", ignore_errors=True)
    zipfile.ZipFile(zip_name).extractall("/content/dataset")
print(os.listdir("/content/dataset"))

In [ ]:
# ===== 셀 2-B: Roboflow API (A 대신 사용) =====
# !pip -q install roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="본인_API_키")
# project = rf.workspace("본인_워크스페이스").project("본인_프로젝트")
# ds = project.version(1).download("yolov8", location="/content/dataset")

In [ ]:
# ===== 셀 3: data.yaml 경로 정리 + 부류 확인 =====
import yaml
DS = "/content/dataset"
EXPECTED = ["giant_hornet", "honeybee", "yellow_legged_hornet"]

def fix_yaml(ds):
    p = os.path.join(ds, "data.yaml")
    d = yaml.safe_load(open(p))
    d["path"] = ds
    for key, sub in [("train", "train"), ("val", "valid"), ("test", "test")]:
        if os.path.isdir(os.path.join(ds, sub, "images")):
            d[key] = f"{sub}/images"
        elif key == "test":
            d.pop("test", None)
    yaml.safe_dump(d, open(p, "w"), allow_unicode=True, sort_keys=False)
    return p, d

DATA, cfg = fix_yaml(DS)
names = cfg["names"] if isinstance(cfg["names"], list) else [cfg["names"][k] for k in sorted(cfg["names"])]
print("부류:", names)
assert names == EXPECTED, f"부류 이름/순서가 다릅니다. Roboflow 라벨 이름을 확인하세요: {names}"
HAS_TEST = "test" in cfg
for sub in ["train", "valid", "test"]:
    d = os.path.join(DS, sub, "images")
    if os.path.isdir(d):
        n_img = len(os.listdir(d))
        n_null = sum(1 for f in os.listdir(os.path.join(DS, sub, "labels"))
                     if os.path.getsize(os.path.join(DS, sub, "labels", f)) == 0)
        print(f"{sub:6s} 이미지 {n_img:4d}장 (박스 없는 이미지 {n_null}장)")

In [ ]:
# ===== 셀 4: 작은 객체 분석 — 입력 해상도별 객체 크기 =====
!python {REPO}/scripts/label_size_stats.py --root {DS} --imgsz 640 960 1280 \
    --names giant_hornet honeybee yellow_legged_hornet --csv {REPO}/results/label_size_stats.csv

## 비교 실험
`RUN`에 넣은 실험만 학습합니다. 핵심 3개(입력 크기 비교)를 먼저 돌리고, 시간이 남으면 나머지를 추가하세요.

| 이름 | 비교 목적 |
| --- | --- |
| n640 / n960 / n1280 | 입력 해상도가 작은 객체(꿀벌) 탐지에 주는 영향 |
| s960 | 모델 크기(n → s)의 영향 |
| n960_hue | 색상 증강을 세게(hsv_h 0.1) 주면 종 구분이 나빠지는지 (기본값 0.015) |
| p2_960 | 4픽셀 간격 격자(P2 헤드)를 추가하면 작은 객체가 나아지는지 |

In [ ]:
# ===== 셀 5: 실험 설정 =====
EPOCHS, PATIENCE, SEED = 50, 10, 0
EXPERIMENTS = {
    "n320":     dict(model="yolov8n.pt", imgsz=320),    # GPU가 없을 때(CPU) 쓰는 가벼운 설정
    "n640":     dict(model="yolov8n.pt", imgsz=640),
    "n960":     dict(model="yolov8n.pt", imgsz=960),
    "n1280":    dict(model="yolov8n.pt", imgsz=1280),
    "s960":     dict(model="yolov8s.pt", imgsz=960),
    "n960_hue": dict(model="yolov8n.pt", imgsz=960, hsv_h=0.1),
    "p2_960":   dict(model="yolov8n-p2.yaml", weights="yolov8n.pt", imgsz=960),
}
RUN = ["n640", "n960"]                   # 먼저 돌릴 실험 (CPU라면 EPOCHS, PATIENCE = 12, 4 와 RUN = ["n320"])

In [ ]:
# ===== 셀 6: 학습 + 테스트 평가 =====
import json, pandas as pd
from ultralytics import YOLO

def train_and_eval(name, cfg):
    cfg = dict(cfg)
    model = YOLO(cfg.pop("model"))
    w = cfg.pop("weights", None)
    if w:
        model = model.load(w)                          # P2 구조에 사전학습 가중치 이식
    imgsz = cfg.pop("imgsz")
    model.train(data=DATA, epochs=EPOCHS, patience=PATIENCE, imgsz=imgsz, batch=-1, seed=SEED,
                project=PROJECT, name=name, exist_ok=True, plots=True, **cfg)
    best = os.path.join(PROJECT, name, "weights", "best.pt")
    m = YOLO(best).val(data=DATA, imgsz=imgsz, split="test" if HAS_TEST else "val",
                       project=PROJECT, name=f"{name}_test", exist_ok=True, plots=True)
    sp = m.speed                                       # 이미지 1장당 ms
    ms = sp["preprocess"] + sp["inference"] + sp["postprocess"]
    row = {"exp": name, "imgsz": imgsz, "precision": m.box.mp, "recall": m.box.mr,
           "mAP50": m.box.map50, "mAP50-95": m.box.map,
           "ms_per_img": ms, "fps": 1000 / ms if ms else float("nan"), "best_pt": best}
    for i, c in enumerate(m.box.ap_class_index):
        row[f"AP50_{names[int(c)]}"] = m.box.ap50[i]
    return row

rows = []
for name in RUN:
    print(f"\n===== {name} =====")
    rows.append(train_and_eval(name, EXPERIMENTS[name]))
    pd.DataFrame(rows).to_csv(os.path.join(PROJECT, "experiments.csv"), index=False)   # 중간 저장

df = pd.DataFrame(rows)
df.round(3)

In [ ]:
# ===== 셀 7: 가장 좋은 모델을 저장소 폴더로 정리 =====
import shutil
METRIC = "mAP50-95"                                  # 선택 기준
best_row = df.sort_values(METRIC, ascending=False).iloc[0]
print("선택:", best_row["exp"], f"({METRIC} {best_row[METRIC]:.3f})")

os.makedirs(f"{REPO}/models", exist_ok=True)
shutil.copy(best_row["best_pt"], f"{REPO}/models/best.pt")
df.drop(columns=["best_pt"]).round(4).to_csv(f"{REPO}/results/experiments.csv", index=False)

for exp in df["exp"]:
    out = f"{REPO}/results/{exp}"
    os.makedirs(out, exist_ok=True)
    for f in ["results.png", "confusion_matrix.png", "confusion_matrix_normalized.png", "BoxPR_curve.png", "PR_curve.png"]:
        for src_dir in [os.path.join(PROJECT, exp), os.path.join(PROJECT, f"{exp}_test")]:
            src = os.path.join(src_dir, f)
            if os.path.exists(src):
                tag = "test_" if src_dir.endswith("_test") else ""
                shutil.copy(src, os.path.join(out, tag + f))
print(sorted(os.listdir(f"{REPO}/results")))

## 분석 (보고서용)
셀 8은 테스트 세트에서 종 혼동·놓침·오탐을 직접 세고 실패 사례 그림을 만듭니다. 셀 9는 자동 라벨링을 검증합니다(선택).

In [ ]:
# ===== 셀 8: 테스트 세트 혼동 분석 + 실패 사례 =====
# 장수말벌 ↔ 등검은말벌을 얼마나 헷갈리는지, 어떤 사진에서 틀리는지 확인합니다 (보고서 '학습 과정 및 성능'용).
import glob, numpy as np
from PIL import Image, ImageDraw
from IPython.display import display
from ultralytics import YOLO

CONF, MATCH_IOU = 0.25, 0.5
best = YOLO(f"{REPO}/models/best.pt")
IMGSZ = int(best_row["imgsz"])
split = "test" if HAS_TEST else "valid"
img_dir, lab_dir = os.path.join(DS, split, "images"), os.path.join(DS, split, "labels")
nc = len(names); BG = nc
cm = np.zeros((nc + 1, nc + 1), int)    # 행 = 정답, 열 = 예측, 마지막 칸 = 배경(박스 없음)
cases = []                               # [파일, 종류, 정답, 예측, 확신도, 정답 박스, 예측 박스]

def box_iou(a, b):
    iw = max(0, min(a[2], b[2]) - max(a[0], b[0])); ih = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)

files_ = sorted(glob.glob(os.path.join(img_dir, "*")))
for res in best.predict(files_, imgsz=IMGSZ, conf=CONF, stream=True, verbose=False):
    f = res.path; H, W = res.orig_shape
    gt = []
    lp = os.path.join(lab_dir, os.path.splitext(os.path.basename(f))[0] + ".txt")
    if os.path.exists(lp):
        for line in open(lp):
            p = line.split()
            if len(p) >= 5:
                c, cx, cy, w, h = int(p[0]), *map(float, p[1:5])
                gt.append((c, ((cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H)))
    pr = [(int(c), tuple(b), float(s)) for c, b, s in zip(res.boxes.cls.tolist(), res.boxes.xyxy.tolist(), res.boxes.conf.tolist())]
    pairs = sorted(((box_iou(g[1], p[1]), i, j) for i, g in enumerate(gt) for j, p in enumerate(pr)), reverse=True)
    ug, up = set(), set()
    for v, i, j in pairs:                # 위치가 겹치는 것끼리 먼저 짝짓고(클래스 무관), 클래스가 같은지 따로 본다
        if v < MATCH_IOU:
            break
        if i in ug or j in up:
            continue
        ug.add(i); up.add(j)
        cm[gt[i][0], pr[j][0]] += 1
        if gt[i][0] != pr[j][0]:
            cases.append([f, "종 혼동", names[gt[i][0]], names[pr[j][0]], pr[j][2], gt[i][1], pr[j][1]])
    for i, g in enumerate(gt):
        if i not in ug:
            cm[g[0], BG] += 1; cases.append([f, "놓침", names[g[0]], "-", 0.0, g[1], None])
    for j, p in enumerate(pr):
        if j not in up:
            cm[BG, p[0]] += 1; cases.append([f, "오탐", "-", names[p[0]], p[2], None, p[1]])

labels_ = names + ["배경"]
cm_df = pd.DataFrame(cm, index=[f"정답:{n}" for n in labels_], columns=[f"예측:{n}" for n in labels_])
print(f"{split} 세트 {len(files_)}장, conf ≥ {CONF}, 짝짓기 IoU ≥ {MATCH_IOU}  (행 = 정답, 열 = 예측)")
display(cm_df)

gi, yi, hi = names.index("giant_hornet"), names.index("yellow_legged_hornet"), names.index("honeybee")
def rate(r, c):
    tot = cm[r, :].sum()
    return f"{cm[r, c]}/{tot} ({cm[r, c] / tot:.1%})" if tot else "-"
print("장수말벌 → 등검은말벌로 오인:", rate(gi, yi))
print("등검은말벌 → 장수말벌로 오인:", rate(yi, gi))
print("꿀벌 → 말벌(두 종)로 오인:", f"{cm[hi, gi] + cm[hi, yi]}/{cm[hi, :].sum()}")
print("놓친 개체(정답 → 배경):", {n: int(cm[k, BG]) for k, n in enumerate(names)})
print("박스 없는 곳에 낸 오탐(배경 → 예측):", {n: int(cm[BG, k]) for k, n in enumerate(names)})

os.makedirs(f"{REPO}/results", exist_ok=True)
cm_df.to_csv(f"{REPO}/results/test_confusion_counts.csv", encoding="utf-8-sig")
pd.DataFrame([c[:5] for c in cases], columns=["file", "type", "true", "pred", "conf"]).to_csv(
    f"{REPO}/results/failure_cases.csv", index=False, encoding="utf-8-sig")

# 실패 사례 그림: 종 혼동 → 확신도 높은 오탐 → 놓침 순으로 최대 16장 (초록 = 정답, 빨강 = 예측)
order = {"종 혼동": 0, "오탐": 1, "놓침": 2}
pick, seen = [], set()
for c in sorted(cases, key=lambda c: (order[c[1]], -c[4])):
    if c[0] not in seen:
        seen.add(c[0]); pick.append(c)
    if len(pick) == 16:
        break
T = 320
sheet = Image.new("RGB", (4 * T, max(1, (len(pick) + 3) // 4) * (T + 24)), "white")
TYPE_EN = {"종 혼동": "CONFUSED", "오탐": "FALSE POS", "놓침": "MISSED"}
for k, c in enumerate(pick):
    im = Image.open(c[0]).convert("RGB"); d = ImageDraw.Draw(im); lw = max(3, im.size[0] // 150)
    for cc in [x for x in cases if x[0] == c[0]]:
        if cc[5] is not None: d.rectangle(cc[5], outline=(0, 200, 0), width=lw)
        if cc[6] is not None: d.rectangle(cc[6], outline=(230, 30, 30), width=lw)
    im.thumbnail((T, T)); x, y = (k % 4) * T, (k // 4) * (T + 24)
    sheet.paste(im, (x, y))
    ImageDraw.Draw(sheet).text((x + 4, y + T + 5), f"{TYPE_EN[c[1]]}  true:{c[2]}  pred:{c[3]}" + (f" {c[4]:.2f}" if c[3] != "-" else ""), fill="black")
sheet.save(f"{REPO}/results/failure_cases.jpg", quality=88)
print(f"실패 사례 {len(cases)}건 중 {len(pick)}장 → results/failure_cases.jpg, 전체 목록 → results/failure_cases.csv")
display(sheet)

In [ ]:
# ===== 셀 9 (선택): 자동 라벨 정확도 — Grounding DINO 박스 vs 검수 후 최종 라벨 =====
# 01b_auto_label.ipynb에서 받은 autolabel_roboflow.zip(필수)과 needs_manual.zip(있으면)을 올립니다.
# 셀 2에서 쓴 Roboflow 내보내기(DS)가 '최종 라벨'입니다. Resize를 "Fit within"이나 끄기로 내보냈어야 정확합니다.
from google.colab import files
import zipfile
up = files.upload()
for name in up:
    target = "/content/needs_manual" if "manual" in name else "/content/autolabel"
    shutil.rmtree(target, ignore_errors=True)
    zipfile.ZipFile(name).extractall(target)
manual_arg = "--manual /content/needs_manual" if os.path.isdir("/content/needs_manual") else ""
!python {REPO}/scripts/compare_autolabel.py --auto /content/autolabel --final {DS} {manual_arg} --csv {REPO}/results/autolabel_eval.csv

In [ ]:
# ===== 셀 10: 결과 묶기 → Drive 저장 + 내려받기 =====
# 03_track.ipynb가 Drive에서 best.pt와 experiments.csv를 가져갑니다.
# train_outputs.zip(models/best.pt + results/)은 저장소에 올릴 파일입니다.
import zipfile
if USE_DRIVE:
    os.makedirs("/content/drive/MyDrive/beeguard", exist_ok=True)
    shutil.copy(f"{REPO}/models/best.pt", "/content/drive/MyDrive/beeguard/best.pt")
    shutil.copy(f"{REPO}/results/experiments.csv", "/content/drive/MyDrive/beeguard/experiments.csv")
with zipfile.ZipFile("/content/train_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write(f"{REPO}/models/best.pt", "models/best.pt")
    for root, _, fs in os.walk(f"{REPO}/results"):
        for f in fs:
            p = os.path.join(root, f)
            z.write(p, os.path.relpath(p, REPO))
print("train_outputs.zip:", os.path.getsize("/content/train_outputs.zip") // 1024, "KB")
from google.colab import files
files.download("/content/train_outputs.zip")

## 다음 단계
- `results/experiments.csv`의 표를 보고서 "학습 과정 및 성능"에 넣습니다. 정확도(mAP)와 속도(FPS)를 함께 적어야 Jetson 적용 판단 근거가 됩니다.
- 셀 8 결과(`results/test_confusion_counts.csv`, `results/failure_cases.jpg`)로 장수말벌 ↔ 등검은말벌 혼동 비율과 실패 사례를 분석합니다. 틀린 사진의 공통점(작은 개체, 역광, 비슷한 종, 라벨 오류 등)을 찾는 게 핵심입니다.
- 셀 9 결과(`results/autolabel_eval.csv`)로 "자동 박스의 정밀도·재현율과 사람이 고친 양"을 보고서에 적습니다.
- 셀 10으로 `train_outputs.zip`을 받아 저장소의 `models/`, `results/`에 넣습니다.
- 추적 영상은 `03_track.ipynb`에서 만듭니다. best.pt는 셀 10이 Drive에 저장한 것을 가져갑니다.